# Analytical training runtime: connected IQP and MLP
Recorded three-seed runtime audit reported in the paper. This notebook only reads saved measurements; it does not train or time new models. The accepted inference-only optimizations preserve the tested trajectories and final states. Runtime is host-specific, and three seeds are not a general benchmark.
Training/audit implementation: `benchmark_optimized_iqp_training.py`. QASP is an external dependency.

In [1]:
from pathlib import Path
import json
from IPython.display import display, Markdown
REPO = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'REPOSITORY_PROVENANCE.json').is_file())
FOLDER = REPO/'results/iqp_simulator_optimization/full_training_inference_only'
summary = json.loads((FOLDER/'summary.json').read_text())
rows = ['| Method | Mean training seconds | Per-seed seconds |', '|---|---:|---|']
for name, s in summary.items():
    values = ', '.join(f'{v:.2f}' for v in s['per_seed_seconds'])
    rows.append(f"| {name} | {s['mean_seconds']:.2f} | {values} |")
display(Markdown('\n'.join(rows)))


| Method | Mean training seconds | Per-seed seconds |
|---|---:|---|
| reference | 43.58 | 46.14, 42.31, 42.29 |
| compiled | 38.45 | 41.05, 39.00, 35.31 |
| compiled_frozen | 34.03 | 35.85, 33.25, 32.99 |
| mlp | 8.98 | 7.84, 9.62, 9.47 |
| mlp_frozen | 7.90 | 6.44, 8.27, 8.98 |

In [2]:
records = [json.loads(p.read_text()) for p in sorted(FOLDER.glob('*__*.json'))]
checks = ['returns_bitwise_equal', 'final_state_bitwise_equal', 'freeze_equal', 'updates_equal']
assert all(all(r[k] for k in checks) for r in records)
print(f'PASS: all four recorded identity checks hold for {len(records)} method/seed records.')
print(json.dumps(json.loads((FOLDER/'protocol.json').read_text()), indent=2))


PASS: all four recorded identity checks hold for 15 method/seed records.
{
  "seeds": [
    2001,
    2002,
    2003
  ],
  "methods": [
    "reference",
    "compiled",
    "compiled_frozen",
    "mlp",
    "mlp_frozen"
  ],
  "episodes": 500,
  "torch": "2.7.0",
  "platform": "macOS-27.0-arm64-arm-64bit",
  "threads": 1,
  "scope": "Training only; compile warmup reported separately. First four methods run serially in rotated order across seeds; MLP frozen-work elimination is a separate three-seed follow-up. Shared desktop, not an exclusive-host benchmark. No changes to circuit, decoder, precision, initialization, loss, optimizer, batching, random streams, freeze rule or total episodes."
}
